## Legal Expert Knowledge Worker

This is a question-answering assistant with access to the Constitution (supreme law), statutory law, English common law, customary law and Islamic law documents. This will make it an expert in legal problems and it will answer any legal questions thrown to it.

### Note: The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

### PART A: Retrieve the documents and Divide them into chunks

The documents will be retrieved from an external source (e.g. Google Drive).

In [1]:
%pip install google-api-python-client google-auth google-auth-oauthlib
%pip install 'markitdown[all]'


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [20]:
import os
from dotenv import load_dotenv

import tiktoken
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go


load_dotenv(override=True)

from google_drive_client import GoogleDriveClient

google_drive_credentials_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')
google_drive_folder_id = os.getenv('GOOGLE_DRIVE_FOLDER_ID')

print(f"Credentials path: {google_drive_credentials_path}")
print(f"Google Drive Folder ID: {google_drive_folder_id}")

# Google Drive client:
collector = GoogleDriveClient(google_drive_credentials_path, google_drive_folder_id)
collector.collect_files()

# Files from a Google Drive folder in markdown format.
documents = collector.get_collection()

# Entire knowledge base
entire_knowledge_base = collector.entire_knowledge_base

print(f"Found {len(documents)} files in the knowledge base")

Credentials path: /home/thomas/Downloads/googleUserContent.json
Google Drive Folder ID: 1dOFomY-wQD0NmBDA1XAH8eUqLaE2A0iz
Total characters in knowledge base: 3,336,279
Found 2 files in the knowledge base


In [ ]:

from openai import OpenAI

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
ds_api_key = os.getenv('DEEPSEEK_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')


OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = openrouter_api_key

# Consider using an enum for this.
MODEL_MAP = {
    "CLAUDE": {
        "model": "claude-3-5-sonnet-20240620",
        "api_key": anthropic_api_key,
        "endpoint": "https://api.anthropic.com/v1"
    },
    "GPT": {
        "model": "gpt-4o-mini",
        "api_key": OPENROUTER_API_KEY,
        "endpoint": OPENROUTER_BASE_URL,
    },
    "Grok": {
        "model": "grok-beta",
        "api_key": grok_api_key,
        "endpoint": "https://api.grok.com/v1"
    },   
    "DeepSeek":{
        "model": "deepseek-reasoner",
        "api_key": ds_api_key,
        "endpoint": "https://api.deepseek.com/v1",
    },
    "Google": {
        "model": "gemini-2.0-flash-exp",
        "api_key": google_api_key,
        "endpoint": "https://generativelanguage.googleapis.com/v1beta/openai"
    },
}


SPECIAL ISSUE
Kenya Gazette Supplement No. 158 (Acts No. 17)
REPUBLIC OF KENYA
–––––––
KENYA GAZETTE SUPPLEMENT
ACTS, 2015
NAIROBI, 15th September, 2015
CONTENT
Act—
PAGE
The Companies Act, 2015.............................................................................267
PRINTED AND PUBLISHED BY THE GOVERNMENT PRINTER, NAIROBI

267
THE COMPANIES ACT
No. 17 of 2015
Date of Assent: 11th September, 2015
Date of Commencement: Section 2 on 15th September, 2015
All other provisions: See Section 1 (3) and (4)
ARRANGEMENT OF SECTIONS
Section
PART I—PRELIMINARY
1—Short title and commencement.
2—Objects of this Act.
3—Interpretation of provisions of this Act.
4—Provisions supplementing definition of “holding
company” in section 3.
PART II—COMPANIES AND COMPANY
FORMATION
Division 1—Types of companies
5—Limited companies.
6—Companies limited by shares.
7—Companies limited by guarantee.
8—Unlimited companies.
9—Private companies.
10—Public companies.
Division 2—Formation and registration of comp

Check the number of tokens.

In [ ]:
# How many tokens in all the documents?
model = MODEL_MAP["GPT"]["model"]

def count_tokens(model, knowledge_base):
    encoding = tiktoken.encoding_for_model(model)
    tokens = encoding.encode(knowledge_base)
    token_count = len(tokens)
    return token_count

token_count = count_tokens(model, entire_knowledge_base)

print(f"Total tokens for {model}: {token_count:,}")

# print(documents['The_Constitution_of_Kenya_2010.md'][0:1000])
# print(documents['companies-act-2015.md'][0:1000])


Total tokens for gpt-4o-mini: 827,763
SPECIAL ISSUE
Kenya Gazette Supplement No. 158 (Acts No. 17)
REPUBLIC OF KENYA
–––––––
KENYA GAZETTE SUPPLEMENT
ACTS, 2015
NAIROBI, 15th September, 2015
CONTENT
Act—
PAGE
The Companies Act, 2015.............................................................................267
PRINTED AND PUBLISHED BY THE GOVERNMENT PRINTER, NAIROBI

267
THE COMPANIES ACT
No. 17 of 2015
Date of Assent: 11th September, 2015
Date of Commencement: Section 2 on 15th September, 2015
All other provisions: See Section 1 (3) and (4)
ARRANGEMENT OF SECTIONS
Section
PART I—PRELIMINARY
1—Short title and commencement.
2—Objects of this Act.
3—Interpretation of provisions of this Act.
4—Provisions supplementing definition of “holding
company” in section 3.
PART II—COMPANIES AND COMPANY
FORMATION
Division 1—Types of companies
5—Limited companies.
6—Companies limited by shares.
7—Companies limited by guarantee.
8—Unlimited companies.
9—Private companies.
10—Public companies.
Divisio

Divide the documents into chunks.

In [24]:
# Divide into chunks using the RecursiveCharacterTextSplitter
# But first convert the documents to a list of langchain documents.
from langchain_core.documents import Document

langchain_documents = [
    Document(page_content=content, metadata={"file_name": filename})
    for filename, content in documents.items()
]

def split_documents(langchain_documents, chunk_size, chunk_overlap):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = text_splitter.split_documents(langchain_documents)
    return chunks

chunks = split_documents(langchain_documents, 1000, 200)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

chunks[100]

Divided into 4359 chunks
First chunk:

page_content='SPECIAL ISSUE
Kenya Gazette Supplement No. 158 (Acts No. 17)
REPUBLIC OF KENYA
–––––––
KENYA GAZETTE SUPPLEMENT
ACTS, 2015
NAIROBI, 15th September, 2015
CONTENT
Act—
PAGE
The Companies Act, 2015.............................................................................267
PRINTED AND PUBLISHED BY THE GOVERNMENT PRINTER, NAIROBI' metadata={'file_name': 'companies-act-2015.md'}


Document(metadata={'file_name': 'companies-act-2015.md'}, page_content='| ---------------- | --- | --- | --- | --------- | --- | ---- | -------- | ---------- | --- |\nstatements with Registrar.\n686—Lodgement requirements for companies subject to\nsmall companies regime.\n687—Lodgement requirements for unquoted companies.\n688—Lodgement requirements for quoted companies.\n| 689—Exemption  |              |     | of  | unlimited  |            |     | companies   | from  |     |\n| -------------- | ------------ | --- | --- | ---------- | ---------- | --- | ----------- | ----- | --- |\n|                | requirement  |     | to  | lodge      | financial  |     | statements  | with  |     |\nRegistrar.\n| 690—Special  |     | auditor´s  |     | report  |     | required  | if  | abbreviated  |     |\n| ------------ | --- | ---------- | --- | ------- | --- | --------- | --- | ------------ | --- |\nfinancial statement is lodged with Registrar.\n| 691—Directors  |     |     | of  | company  |  

### PART B: Make vectors and store in Chroma

In [25]:
# Pick an embedding model
db_name = "vector_db"
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 4359 documents


In [26]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 4,359 vectors with 384 dimensions in the vector store


### Part C: Visualize!

This section visualizes the document embeddings so we can inspect how chunks cluster by source document.

- Fetches embeddings, documents, and metadata from the Chroma collection via `collection.get()`; converts embeddings to a NumPy array; derives `sources` from `file_name` metadata; assigns `colors` by document (red = Constitution, blue = Companies Act) 
- Applies **t-SNE** with `n_components=2` to reduce 384‑dim vectors to 2D; builds an interactive Plotly `go.Scatter` plot; hover shows source and text preview; layout 800×600 
- Same pipeline with **t-SNE** `n_components=3`; uses `go.Scatter3d` for a rotatable 3D scatter plot; layout 900×700 with x/y/z axes titled.


In [34]:
# Prework
import numpy as np

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
sources = [metadata['file_name'] for metadata in metadatas]
colors = [['red', 'blue'][['The_Constitution_of_Kenya_2010.md', 'companies-act-2015.md'].index(t)] for t in sources]

In [35]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(sources, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [37]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(sources, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.
Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [38]:
# Connect to Chroma; use Hugging Face all-MiniLM-L6-v2
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm


In [52]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

base_url = MODEL_MAP["GPT"]["endpoint"]
api_key = MODEL_MAP["GPT"]["api_key"]

MODEL = "gpt-4.1-nano"

retriever = vectorstore.as_retriever()
llm = ChatOpenAI(
    base_url=base_url,
    api_key=api_key,
    temperature=0, model_name=MODEL)

Retriever

In [41]:
retriever.invoke("What is the financial year of a company?")

[Document(id='b8070fb7-1bad-4366-9e5b-040c6824c353', metadata={'file_name': 'companies-act-2015.md'}, page_content='financial year of the company—a report stating\n|     | without  |     | material  |     | qualification  |     | the  | auditor´s  |     |\n| --- | -------- | --- | --------- | --- | -------------- | --- | ---- | ---------- | --- |'),
 Document(id='6d847f6e-d94d-4b5b-a5ea-a1177d7f42c1', metadata={'file_name': 'companies-act-2015.md'}, page_content='of the financial year to which its financial statement relates.\n627. For the purposes of this Part, a company is a  When company is a\nquoted company or\nquoted company in relation to a financial year if it was a  an unquoted\nquoted  company  immediately  before  the  end  of  the  company for the\npurposes of this Part.\n| accounting  |     | reference  | period  | by  reference  |     | to  which  | that  |     |\n| ----------- | --- | ---------- | ------- | -------------- | --- | ---------- | ----- | --- |\nfinancial year

LLM

In [44]:
llm.invoke("What is the financial year of a company?")

AIMessage(content="The financial year of a company, also known as the fiscal year, is a 12-month period used for accounting and financial reporting purposes. It may or may not align with the calendar year (January 1 to December 31). The specific start and end dates of a company's financial year are determined by the company itself and can vary depending on the country, industry, or internal policies.\n\nFor example:\n- A company might have a financial year from April 1 to March 31.\n- Another might have a financial year from July 1 to June 30.\n\nThe financial year is important for preparing financial statements, tax filings, and analyzing the company's financial performance over a consistent period.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 139, 'prompt_tokens': 16, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'ima

In [46]:
SYSTEM_PROMPT_TEMPLATE = """
You are a legal expert knowledgeable or conversant with Kenyan company law. 
You are chatting with a user about Kenyan company law.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.

Context:
{context}
"""

In [47]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [51]:
answer_question("What does the law say about leadership in a company?", [])


"In Kenyan company law, leadership within a company is primarily governed by the duties and responsibilities of directors, as outlined in the Companies Act, 2015. The law emphasizes that directors have a fiduciary duty to act in the best interests of the company and to comply with applicable legal and regulatory requirements.\n\nSpecifically, the law stipulates that:\n\n1. **General Duties of Directors**: Directors must act honestly, in good faith, and in the best interests of the company. They are expected to exercise their powers for a proper purpose and to avoid conflicts of interest.\n\n2. **Legal and Fiduciary Responsibilities**: Directors are bound by fiduciary duties, which include avoiding conflicts of interest, not making secret profits, and exercising care, diligence, and skill in managing the company.\n\n3. **Accountability and Oversight**: Leadership involves accountability to shareholders and other stakeholders. Directors are responsible for the management and strategic di

In [54]:
gr.ChatInterface(answer_question).launch()

/home/thomas/Desktop/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning:

The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.



* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
